# TP Ciencia de Datos 2026 — Actividad 1
## Scraping tradicional de `quotes.toscrape.com`

**Objetivo**: armar un dataset de **25 citas** con `cita`, `autor` y `tags` (separados por `-`).

**Restricción**: descartar las frases cuyos autores sean **Albert Einstein** o **Thomas A. Edison**.

**Sitio**: <https://quotes.toscrape.com> — HTML estático, sin JavaScript dinámico, ideal para `requests` + `BeautifulSoup`.

## Pseudocódigo / paso a paso

Antes de programar, conviene pensar el algoritmo de manera estructurada:

```
1. registros ← []
2. url       ← página 1 del sitio
3. mientras url existe Y len(registros) < 25:
       a. descargar HTML
       b. parsear con BeautifulSoup
       c. para cada bloque <div class="quote">:
            - extraer texto, autor, tags
            - si autor está en {Einstein, Edison} → saltar
            - si len(registros) == 25 → cortar
            - agregar a registros con tags unidos por '-'
       d. buscar el botón Next; si existe, url ← siguiente página
4. exportar registros a CSV
```

## Decisiones técnicas

| Decisión | Por qué |
|---|---|
| `requests` + `BeautifulSoup` | El sitio es HTML estático. No hace falta Selenium ni Playwright (más pesados). |
| Parser `html.parser` | Viene incluido en la stdlib de Python, sin compilar nada extra (a diferencia de `lxml`). |
| Detección dinámica del botón **Next** | Más robusto que asumir N páginas fijas. Si el sitio agrega/quita páginas, el código sigue funcionando. |
| Corte temprano al llegar a 25 | Evita descargar páginas innecesarias. |
| `set` para autores excluidos | Lookup en O(1) en vez de O(n) con una lista. |
| Tags unidos por `-` | Lo pide el enunciado. |
| `encoding='utf-8-sig'` al exportar | Excel en Windows interpreta bien tildes y comillas tipográficas con el BOM. |

## 1. Importación de librerías

In [8]:
import requests                    # cliente HTTP simple y robusto
from bs4 import BeautifulSoup       # parseo de HTML mediante selectores CSS
import pandas as pd                  # exportación a CSV con una línea
import time                          # pausas de cortesía entre requests

## 2. Configuración: constantes

Definimos las constantes en un solo lugar. Si mañana cambia el dominio o la lista de exclusión, solo se edita acá.

In [9]:
BASE_URL = 'https://quotes.toscrape.com'
AUTORES_EXCLUIDOS = {'Albert Einstein', 'Thomas A. Edison'}
LIMITE_REGISTROS = 25

## 3. Función principal de scraping

Recorre páginas hasta cumplir el límite. Por cada `<div class="quote">`:

1. Extrae el texto de la cita (`<span class="text">`).
2. Extrae el autor (`<small class="author">`).
3. Si el autor está excluido, salta sin agregar.
4. Limpia las comillas tipográficas del texto ("cita textual sin comillas" del enunciado).
5. Recolecta todos los tags (`<a class="tag">`) y los une con `-`.

**Detalle defensivo**: `select_one()` devuelve `None` si no encuentra el elemento. Guardamos el resultado en una variable y usamos `if tag else ""` para evitar `AttributeError`.

In [10]:
def scrapear_citas() -> list[dict]:
    registros: list[dict] = []
    url = BASE_URL + '/page/1/'

    while url and len(registros) < LIMITE_REGISTROS:
        print(f'Scrapeando: {url}  (acumulados: {len(registros)}/{LIMITE_REGISTROS})')

        # timeout=10s evita que el script quede colgado si el server no responde.
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()  # falla rápido ante 4xx/5xx
        soup = BeautifulSoup(resp.text, 'html.parser')

        for bloque in soup.select('div.quote'):
            if len(registros) >= LIMITE_REGISTROS:
                break

            tag_texto = bloque.select_one('span.text')
            tag_autor = bloque.select_one('small.author')
            texto = tag_texto.get_text(strip=True) if tag_texto else ''
            autor = tag_autor.get_text(strip=True) if tag_autor else ''

            # FILTRO: descartamos a Einstein y Edison.
            if autor in AUTORES_EXCLUIDOS:
                continue

            # Quitamos comillas tipográficas “ ” y rectas " '
            texto = texto.strip('\u201c\u201d"\'')

            tags = [a.get_text(strip=True) for a in bloque.select('a.tag')]
            tags_str = '-'.join(tags)

            registros.append({'cita': texto, 'autor': autor, 'tags': tags_str})

        # Paginación: buscamos <li class="next"><a href="/page/N/">
        next_btn = soup.select_one('li.next > a')
        url = BASE_URL + str(next_btn['href']) if next_btn else None

        time.sleep(0.3)  # pausa de cortesía con el servidor

    return registros

## 4. Ejecución

Llamamos la función y verificamos cuántos registros obtuvimos.

In [11]:
registros = scrapear_citas()
print(f'\nTotal de registros obtenidos: {len(registros)}')

Scrapeando: https://quotes.toscrape.com/page/1/  (acumulados: 0/25)
Scrapeando: https://quotes.toscrape.com/page/2/  (acumulados: 6/25)
Scrapeando: https://quotes.toscrape.com/page/3/  (acumulados: 15/25)
Scrapeando: https://quotes.toscrape.com/page/4/  (acumulados: 23/25)

Total de registros obtenidos: 25


## 5. Construcción del DataFrame y exportación

Convertimos la lista de diccionarios en un `DataFrame` de pandas para inspección rápida y exportamos a CSV.

In [12]:
df = pd.DataFrame(registros)
df.head()

,cita,autor,tags
0,"It is our choices, Harry, that show what we tr...",J.K. Rowling,abilities-choices
1,"The person, be it gentleman or lady, who has n...",Jane Austen,aliteracy-books-classic-humor
2,"Imperfection is beauty, madness is genius and ...",Marilyn Monroe,be-yourself-inspirational
3,It is better to be hated for what you are than...,André Gide,life-love
4,A woman is like a tea bag; you never know how ...,Eleanor Roosevelt,misattributed-eleanor-roosevelt


In [13]:
df.to_csv('actividad_1_citas.csv', index=False, encoding='utf-8-sig')
print(f'✔ Dataset guardado: actividad_1_citas.csv ({len(df)} registros)')

✔ Dataset guardado: actividad_1_citas.csv (25 registros)


## 6. Validación rápida

Verificamos que no haya valores nulos y que ningún autor excluido se haya colado.

In [14]:
print('Nulos por columna:')
print(df.isna().sum())

print('\n¿Algún autor excluido en el dataset?')
print(df[df['autor'].isin(AUTORES_EXCLUIDOS)])

print('\nDistribución de autores:')
print(df['autor'].value_counts())

Nulos por columna:
cita     0
autor    0
tags     0
dtype: int64

¿Algún autor excluido en el dataset?
Empty DataFrame
Columns: [cita, autor, tags]
Index: []

Distribución de autores:
autor
J.K. Rowling           4
Dr. Seuss              3
Marilyn Monroe         2
Bob Marley             2
Jane Austen            1
André Gide             1
Eleanor Roosevelt      1
Steve Martin           1
Douglas Adams          1
Elie Wiesel            1
Friedrich Nietzsche    1
Mark Twain             1
Allen Saunders         1
Pablo Neruda           1
Ralph Waldo Emerson    1
Mother Teresa          1
Garrison Keillor       1
Jim Henson             1
Name: count, dtype: int64
